In [56]:
from utils import readJson, writeJson
import os
import unicodedata
from thefuzz import fuzz, process
import pandas as pd

In [ ]:
def remove_accents(text: str) -> str:
    char_map = {
        'æ': 'ae', 'Æ': 'Ae',
        'ø': 'o', 'Ø': 'O',
        'å': 'a', 'Å': 'A'
    }
    normalized_text = unicodedata.normalize('NFKD', text)
    return ''.join(
        char_map.get(c, c) if unicodedata.category(c) != 'Mn' else ''
        for c in normalized_text
    )

def merge_teams(set1, set2, threshold=50):
    """Accoppia squadre con nomi simili tra due set."""
    merged_teams = {}
    
    for team in set1:
        match, score = process.extractOne(team, set2)
        if score >= threshold:
            merged_teams[team] = match
        else:
            merged_teams[team] = None  # Nessuna corrispondenza trovata
    
    return merged_teams


In [20]:
path_fbref = 'Dataset/Fbref'
path_transfermarkt = 'Dataset/Transfermarkt/Rosa'

In [22]:
leagues = [x for x in os.listdir(path_transfermarkt) if x.endswith('2023.json')]
leagues = [x for x in leagues if not x.startswith('eredivisie') and not x.startswith('ligaportugal') ]
leagues

['bundesliga_2023.json',
 'laliga_2023.json',
 'ligue1_2023.json',
 'premierleague_2023.json',
 'seriea_2023.json']

In [111]:
rose_tm = []
for l in leagues:
    rose_l = readJson(f'{path_transfermarkt}/{l}')
    rose_tm = rose_tm + rose_l
rose_fb = readJson(f'{path_fbref}/players.json')

In [ ]:
fbref_teams = set([x['team'] for x in rose_fb])
tm_teams = set([x['team'] for x in rose_tm])
merged = merge_teams(tm_teams, fbref_teams)
#writeJson(merged, 'Dataset/merged_teams.json')

In [49]:
teams_dict = readJson("Dataset/merged_teams.json")

In [112]:
tm_keys = list(rose_tm[0].keys())
for p_tm in rose_tm:
    for key in tm_keys:
        p_tm[f'tm_{key}'] = p_tm.pop(key)

rose_tm[0]

{'tm_shirt_number': '1',
 'tm_id': '17259',
 'tm_name': 'Manuel Neuer ',
 'tm_role': 'Portiere',
 'tm_birth_date': '27/mar/1986',
 'tm_nationality': 'Germania',
 'tm_market_value': '4,00 mln €',
 'tm_team': 'FC Bayern Monaco'}

In [162]:
rose_tm = []
for l in leagues:
    rose_l = readJson(f'{path_transfermarkt}/{l}')
    rose_tm = rose_tm + rose_l
rose_fb = readJson(f'{path_fbref}/players.json')
tm_keys = list(rose_tm[0].keys())
rose_merged = []
tresholds = [95, 90, 85, 80]
tm_keys = list(rose_tm[0].keys())
for p_tm in rose_tm:
    for key in tm_keys:
        p_tm[f'tm_{key}'] = p_tm.pop(key)
for tm, fb in teams_dict.items():
    i=0
    j=0
    for p in rose_fb:
        if p['team'] == fb:
            for p_tm in rose_tm:
                if p_tm['tm_team'] == tm:
                    match, score = process.extractOne(p['name'], [p_tm['tm_name']], scorer=fuzz.token_set_ratio)
                    if 'match_score' not in p.keys():
                        for t in tresholds:
                            if score >= t:
                                new_p = p | p_tm
                                p['match_score'] = score
                                j+=1
                                break
                    elif score > p['match_score']:
                                p['match_score'] = score
                                new_p = p | p_tm
            rose_merged.append(new_p)
            i+=1
    '''
    if j > i:
        print(f'{tm} + di {fb}')
    elif i > j:
        print(f'{tm} - di {fb}')
    '''

In [170]:
writeJson(rose_merged, 'Dataset/transfermarkt_fbref_dataset.json')

In [169]:
j=0
for r in rose_merged:
    if r['team'] == 'Frosinone':
        print(r['name'],r['tm_name'])
        j+=1


Matias Soule Matías Soulé
Caleb Okoli Caleb Okoli
Enzo Barrenechea Enzo Barrenechea
Stefano Turati Stefano Turati
Simone Romagnoli Simone Romagnoli
Luca Mazzitelli Luca Mazzitelli 
Marco Brescianini Marco Brescianini
Walid Cheddira Walid Cheddira
Pol Lirola Pol Lirola
Francesco Gelli Francesco Gelli
Ilario Monterisi Ilario Monterisi
Nadir Zortea Nadir Zortea
Anthony Oyono Anthony Oyono
Emanuele Valeri Emanuele Valeri
Riccardo Marchizza Riccardo Marchizza
Reinier Reinier
Abdou Harroui Abdou Harroui
Kaio Jorge Kaio Jorge
Michele Cerofolini Michele Cerofolini
Marvin Cuni Marvin Cuni
Jaime Baez Jaime Báez
Luca Garritano Luca Garritano
Arijon Ibrahimovic Arijon Ibrahimovic
Kevin Bonifazi Kevin Bonifazi
Demba Seck Demba Seck
Mehdi Bourabia Mehdi Bourabia
Giuseppe Caso Giuseppe Caso
Fares Ghedjemis Farès Ghedjemis


In [167]:
len(rose_merged)

2306

In [165]:
k=0
for r in rose_fb:
    if r['team'] == 'Milan':
        k+=1
        print(r['name'], r['tm_name'])
k

KeyError: 'tm_name'

In [160]:
process.extractOne("Lucas Beltran", ["Lucas Beltràn", "Lucas Martínez Quarta"], scorer=fuzz.token_set_ratio)
process.extractOne("Yann Aurel Bisseck", ["Yann Sommer", "Yann Bisseck"], scorer=fuzz.token_set_ratio)
#process.extractOne("Imanol",["Álex Berenguer","Imanol García de Albéniz","Iker Muniain"], scorer=fuzz.token_set_ratio)

('Yann Bisseck', 100)

In [104]:
rose_merged[300]

{'id': '2cfb6e88',
 'name': 'Hugo Gonzalez',
 'link': 'https://fbref.com/en/players/2cfb6e88/scout/12202/Hugo-Gonzalez-Scouting-Report',
 'team': 'Valencia',
 'stats': {},
 'tm_shirt_number': '-',
 'tm_id': '707279',
 'tm_name': 'Hugo González',
 'tm_role': 'Ala destra',
 'tm_birth_date': '07/feb/2003',
 'tm_nationality': 'Spagna',
 'tm_market_value': '300 mila €',
 'tm_team': 'Valencia CF'}